# Bollinger Band Bounce on SPY
## Strategy Brief
The Bollinger Band Bounce strategy is a mean-reversion trading approach that uses Bollinger Bands to identify potential buy and sell signals. When the price of SPY touches or moves outside the lower Bollinger Band, it is considered oversold, suggesting a buy signal. Conversely, when the price touches or moves outside the upper Bollinger Band, it is considered overbought, suggesting a sell signal. This strategy aims to capitalize on the tendency of prices to revert to the mean, or the middle band. Historical testing of this strategy can reveal its effectiveness and potential profitability.
## References
- (No external references)

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

## PHASE 1 - Trading Context
In this phase, we define the parameters for the Bollinger Band Bounce strategy. These parameters include the lookback period for the Bollinger Bands and the number of standard deviations for the bands.

In [ ]:
LOOKBACK_PERIOD = 20
NUM_STD_DEV = 2
START_DATE = '2010-01-01'
END_DATE = '2023-10-01'

## PHASE 2 - Data Exploration
We will download historical data for SPY from Yahoo Finance, calculate the Bollinger Bands, and plot them along with the price data to visualize potential trading signals.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Download SPY data
data = yf.download('SPY', start=START_DATE, end=END_DATE)

# Calculate Bollinger Bands
rolling_mean = data['Close'].rolling(window=LOOKBACK_PERIOD).mean()
rolling_std = data['Close'].rolling(window=LOOKBACK_PERIOD).std()
data['Upper Band'] = rolling_mean + (rolling_std * NUM_STD_DEV)
data['Lower Band'] = rolling_mean - (rolling_std * NUM_STD_DEV)

# Plot
plt.figure(figsize=(14, 7))
plt.plot(data['Close'], label='SPY Close')
plt.plot(rolling_mean, label='Middle Band', color='orange')
plt.plot(data['Upper Band'], label='Upper Band', color='green')
plt.plot(data['Lower Band'], label='Lower Band', color='red')
plt.title('SPY with Bollinger Bands')
plt.legend()
plt.show()

## PHASE 3 - Strategy Engineering
We will create a signal series based on the Bollinger Bands. A buy signal is generated when the price touches the lower band, and a sell signal is generated when the price touches the upper band.

In [ ]:
# Generate signals
data['Signal'] = 0
# Buy signal
data.loc[data['Close'] < data['Lower Band'], 'Signal'] = 1
# Sell signal
data.loc[data['Close'] > data['Upper Band'], 'Signal'] = -1

# Create positions based on signals
data['Position'] = data['Signal'].shift(1).fillna(0)

## PHASE 4 - Coding & Backtesting
We will calculate daily returns based on the positions and plot the equity curve to visualize the strategy's performance.

In [ ]:
# Calculate daily returns
data['Market Returns'] = data['Close'].pct_change()
data['Strategy Returns'] = data['Market Returns'] * data['Position']

# Calculate equity curve
data['Equity Curve'] = (1 + data['Strategy Returns']).cumprod()

# Plot equity curve
plt.figure(figsize=(14, 7))
plt.plot(data['Equity Curve'], label='Equity Curve')
plt.title('Equity Curve of Bollinger Band Bounce Strategy')
plt.legend()
plt.show()

## PHASE 5 - Performance Evaluation
We will evaluate the strategy using various performance metrics such as CAGR, Sharpe Ratio, Sortino Ratio, Calmar Ratio, and maximum drawdown. We will also compare these metrics to a buy-and-hold strategy.

In [ ]:
def calculate_performance_metrics(df):
    # CAGR
    total_return = df['Equity Curve'].iloc[-1] - 1
    num_years = (df.index[-1] - df.index[0]).days / 365.25
    cagr = (1 + total_return) ** (1 / num_years) - 1
    
    # Sharpe Ratio
    sharpe_ratio = df['Strategy Returns'].mean() / df['Strategy Returns'].std() * np.sqrt(252)
    
    # Sortino Ratio
    downside_std = df[df['Strategy Returns'] < 0]['Strategy Returns'].std()
    sortino_ratio = df['Strategy Returns'].mean() / downside_std * np.sqrt(252)
    
    # Calmar Ratio
    max_drawdown = ((df['Equity Curve'].cummax() - df['Equity Curve']).max())
    calmar_ratio = cagr / max_drawdown
    
    # Buy and Hold
    buy_and_hold_return = df['Market Returns'].cumsum().iloc[-1]
    
    return {
        'CAGR': cagr,
        'Sharpe Ratio': sharpe_ratio,
        'Sortino Ratio': sortino_ratio,
        'Calmar Ratio': calmar_ratio,
        'Max Drawdown': max_drawdown,
        'Buy and Hold Return': buy_and_hold_return
    }

# Calculate metrics
metrics = calculate_performance_metrics(data)

# Display metrics
metrics

## PHASE 6 - Deploy & Monitor
We will create a function that downloads the last 60 days of SPY data, computes today's signal, and prints the suggested position.

In [ ]:
def get_latest_signal():
    # Download last 60 days of SPY data
    recent_data = yf.download('SPY', period='60d')
    
    # Calculate Bollinger Bands
    recent_mean = recent_data['Close'].rolling(window=LOOKBACK_PERIOD).mean()
    recent_std = recent_data['Close'].rolling(window=LOOKBACK_PERIOD).std()
    recent_data['Upper Band'] = recent_mean + (recent_std * NUM_STD_DEV)
    recent_data['Lower Band'] = recent_mean - (recent_std * NUM_STD_DEV)
    
    # Determine the latest signal
    latest_close = recent_data['Close'].iloc[-1]
    lower_band = recent_data['Lower Band'].iloc[-1]
    upper_band = recent_data['Upper Band'].iloc[-1]
    
    if latest_close < lower_band:
        signal = 'Buy'
    elif latest_close > upper_band:
        signal = 'Sell'
    else:
        signal = 'Hold'
    
    print(f"Today's signal for SPY is: {signal}")

# Get the latest signal
get_latest_signal()